# 06 - Output Parquet
Grava `gold_partidas_clima` em Parquet + Snappy, particionado por `ano/mes`, e copia para `output/` do repositório.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import logging
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)

from src.config import load_settings
from src.load.writer import copy_to_repo, write_parquet

os.environ["FOOTBALL_DATA_API_KEY"] = dbutils.secrets.get(scope="fc", key="football_data_api_key")
settings = load_settings()

In [0]:
caminho = write_parquet(spark, settings)
print("Gravado em:", caminho)
display(dbutils.fs.ls(f"{caminho}/ano=2026"))

In [0]:
import pyarrow.parquet as pq
from pathlib import Path

lido = spark.read.parquet(caminho)
print("Linhas lidas do Parquet:", lido.count())
display(lido.groupBy("ano", "mes").count().orderBy("ano", "mes"))

arquivo = next(Path(caminho).rglob("*.parquet"))
print("Exemplo de arquivo:", arquivo.name)
print("Compressão:", pq.ParquetFile(arquivo).metadata.row_group(0).column(0).compression)

In [0]:
so_marco = spark.read.parquet(caminho).filter("ano = 2026 AND mes = 3")
print("Jogos em março:", so_marco.count())
so_marco.explain(mode="formatted")

In [0]:
copiados = copy_to_repo(caminho)
print(f"{len(copiados)} arquivo(s) copiado(s) para output/:")
for arquivo in copiados:
    print(" ", arquivo.relative_to(arquivo.parents[3]))